# KernelPack RAG — Generation Baseline (v1)

Wire retrieval → generation and expose a stub eval loop for the PhD validation framework.

## What this notebook does

| Section | Purpose |
|---------|---------|
| Setup | Deps, paths, Qdrant client, BM25 |
| Retrieval | `retrieve_qdrant_hybrid` — Qdrant-native (no ChromaDB) |
| Parity check | Assert recall@3/5/10 matches notebook 03 before proceeding |
| Generation | `generate()` — OpenAI API call with retrieved chunks in prompt |
| Smoke test | One query end-to-end |
| Eval loop | `run_eval_loop()` — drop PhD task JSON in, get output files out |
| Validation stub | `validate()` — replace body when Matt/Ram's scripts arrive |

## Prerequisite

Notebook 03 must have been run at least once so `qdrant_storage/` and `chroma_export.pkl` exist.
Collection name and storage path must match those set in 03.


In [29]:
qdrant.close()


In [16]:
%pip install -q \
    qdrant-client \
    sentence-transformers==5.5.0 \
    rank-bm25==0.2.2 \
    openai


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import json
import os
import re
import sys
from pathlib import Path

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from qdrant_client import QdrantClient
import openai

# utils carried forward from 02b / 03
# (utils/ must be on the path — same repo layout as prior notebooks)

In [18]:
ROOT = Path.cwd()
if (ROOT / "experiments").exists():
    EXPERIMENTS_DIR = ROOT / "experiments"
else:
    EXPERIMENTS_DIR = ROOT
    ROOT = EXPERIMENTS_DIR.parent

if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

EXPORT_PATH        = EXPERIMENTS_DIR / "chroma_export.pkl"
QDRANT_STORAGE     = ROOT / "qdrant_storage"
COLLECTION_NAME    = "kernelpack-v3"
QA_PATH            = EXPERIMENTS_DIR / "qa_pairs" / "solvers_qa.json"

# where PhD task prompts land — drop tasks.json here when they share it
TASKS_PATH         = EXPERIMENTS_DIR / "eval_tasks" / "tasks.json"
# where generated code files are written for the validation script
OUTPUT_DIR         = EXPERIMENTS_DIR / "eval_outputs"

print(f"Experiments : {EXPERIMENTS_DIR}")
print(f"Qdrant store: {QDRANT_STORAGE}")
print(f"Export      : {EXPORT_PATH}")
assert EXPORT_PATH.exists(),    "chroma_export.pkl not found — run notebook 03 first"
assert QDRANT_STORAGE.exists(), "qdrant_storage/ not found — run notebook 03 first"

Experiments : /Users/jordanchambers/public-projects/scientific-codebase-rag/experiments
Qdrant store: /Users/jordanchambers/public-projects/scientific-codebase-rag/qdrant_storage
Export      : /Users/jordanchambers/public-projects/scientific-codebase-rag/experiments/chroma_export.pkl


## Load Infrastructure

Same model, export, and Qdrant client as notebook 03. BM25 is rebuilt from the export each run —
same behaviour as 03, no persistence needed.

In [19]:
from scripts.migrate_to_qdrant import load_chroma_export, make_qdrant_client, stable_point_id
from utils.chunking import bm25_tokenize
from utils.eval_utils import is_hit

# embedding model — must match the model used during migration
model = SentenceTransformer("microsoft/unixcoder-base")
model.max_seq_length = 512
ef = SentenceTransformerEmbeddingFunction(model_name="microsoft/unixcoder-base")
ef._model = model

# Qdrant (persistent — same storage as 03)
qdrant = make_qdrant_client(storage_path=QDRANT_STORAGE)
collection_info = qdrant.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}': {collection_info.points_count} points")

# ChromaDB export — used only for BM25 (document text and IDs)
export = load_chroma_export(EXPORT_PATH)
bm25   = BM25Okapi([bm25_tokenize(doc) for doc in export.documents])
print(f"BM25 index  : {len(export.ids)} documents")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 58425.49it/s]


Collection 'kernelpack-v3': 187 points
BM25 index  : 187 documents


## Retrieval

`retrieve_qdrant_hybrid` mirrors notebook 03 but is **ChromaDB-free** — document text is
fetched from the Qdrant payload (`payload["text"]`) rather than from a rehydrated ChromaDB
collection. Everything else (BM25 tokenizer, RRF k=60, candidate pool size) is identical.

In [24]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10731.83it/s]


Reranker loaded


In [25]:
def embed_query(query: str) -> list[float]:
    """Embed a query string using the same function as 03."""    
    return ef([query])[0]

def retrieve_qdrant_hybrid_reranked(query: str, k: int = 10, fetch_k: int = 30) -> list[dict]:
    """Qdrant hybrid + cross-encoder reranker. fetch_k candidates from hybrid, rerank to k."""
    candidates = retrieve_qdrant_hybrid(query, k=fetch_k)  # ← hybrid, not dense
    pairs = [(query, candidate["text"]) for candidate in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda item: item[0], reverse=True)

    results = []
    for score, candidate in ranked[:k]:
        payload = dict(candidate)
        payload["score"] = float(score)
        payload["score_kind"] = "cross-encoder score; higher is better"
        results.append(payload)
    return results


def retrieve_qdrant_hybrid(query: str, k: int = 10) -> list[dict]:
    """Hybrid BM25 + dense retrieval with RRF fusion.

    Qdrant-native: no ChromaDB dependency. Text fetched from payload["text"].

    Args:
        query: natural language or code query string.
        k:     number of results to return.

    Returns:
        List of payload dicts ordered by RRF score descending.
        Each dict contains: text, function_name, class_name, file_path,
        start_line, end_line, original_id, score, score_kind.
    """
    query_vector = embed_query(query)

    # dense leg — top-10 by cosine similarity
    dense_hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=10,
    ).points
    dense_ids = [h.payload["original_id"] for h in dense_hits]

    # BM25 leg — top-10 by keyword overlap
    bm25_scores = bm25.get_scores(bm25_tokenize(query))
    top_bm25    = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    bm25_ids    = [export.ids[i] for i in top_bm25]

    # RRF fusion — k=60 smoothing constant (matches 02b and 03)
    rrf: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    top_orig_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]

    # fetch text + metadata from Qdrant payload (no ChromaDB)
    point_ids = [stable_point_id(oid) for oid in top_orig_ids]
    points    = qdrant.retrieve(
        collection_name=COLLECTION_NAME,
        ids=point_ids,
        with_payload=True,
    )
    point_map = {p.id: p for p in points}

    results = []
    for orig_id, pt_id in zip(top_orig_ids, point_ids):
        point = point_map.get(pt_id)
        if point is None:
            continue
        payload = dict(point.payload)
        payload["score"]      = rrf[orig_id]
        payload["score_kind"] = "RRF score; higher is better"
        results.append(payload)

    return results


print("retrieve_qdrant_hybrid defined.")

retrieve_qdrant_hybrid defined.


## Parity Check

Run the same recall@k eval as notebook 03 and assert numbers match.
**This cell must pass before running generation.** If it fails, the retrieval
function above has diverged from 03 — do not proceed.

Expected (from notebook 03):
- recall@3  = 8/10 (80%)
- recall@5  = 10/10 (100%)
- recall@10 = 10/10 (100%)

In [21]:
with open(QA_PATH) as f:
    qa_pairs_solvers = json.load(f)

print(f"QA pairs loaded: {len(qa_pairs_solvers)}")

EXPECTED = {3: 8, 5: 10, 10: 10}

for k, expected_hits in EXPECTED.items():
    hits = sum(
        1 for qa in qa_pairs_solvers
        if any(is_hit(r, qa["source_symbol"]) for r in retrieve_qdrant_hybrid(qa["query"], k=k))
    )
    status = "OK" if hits == expected_hits else f"MISMATCH — expected {expected_hits}"
    print(f"recall@{k:2d}: {hits}/{len(qa_pairs_solvers)}  {status}")
    assert hits == expected_hits, (
        f"Parity check failed at k={k}: got {hits}, expected {expected_hits}. "
        "Retrieval has diverged from notebook 03. Do not proceed."
    )

print("\nParity check passed — retrieval matches notebook 03.")

QA pairs loaded: 10
recall@ 3: 8/10  OK
recall@ 5: 10/10  OK
recall@10: 10/10  OK

Parity check passed — retrieval matches notebook 03.


## Generation

`generate()` calls the OpenAI API with retrieved chunks stuffed into the prompt.
Set `OPENAI_API_KEY` in your environment before running.

**Model choice:** `gpt-4o` is the default. Swap to `o3` for stronger code reasoning
at higher cost/latency.

In [27]:
# ── API key ────────────────────────────────────────────────────────────────────
from openai import OpenAI
client = OpenAI()
print("OpenAI client ready.")

GENERATION_MODEL = "gpt-4o"   # swap to "o3" for stronger code reasoning

SYSTEM_PROMPT = """You are assisting with scientific computing code that uses the KernelPack library.
Use ONLY the following retrieved source context to answer.
Do not invent function names, class names, or argument names not present in the context below.
If the retrieved context is insufficient to complete the task, say so explicitly rather than guessing."""


def format_chunks_for_prompt(chunks: list[dict], max_chars: int = 8000) -> str:
    """Format retrieved chunks as a context block for the LLM prompt.

    Truncates at max_chars to avoid blowing the context window.
    Each chunk is labelled with its symbol name and file path.
    """
    parts = []
    total = 0
    for i, chunk in enumerate(chunks, 1):
        fn   = chunk.get("function_name") or "<module>"
        cls  = chunk.get("class_name") or ""
        sym  = f"{cls}.{fn}" if cls else fn
        path = chunk.get("file_path") or chunk.get("source_file") or ""
        text = chunk.get("text", "")
        block = f"--- Chunk {i} | {sym} | {path} ---\n{text}"
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "\n\n".join(parts)


def generate(task_prompt: str, chunks: list[dict], model: str = GENERATION_MODEL) -> str:
    """Call the OpenAI API with retrieved chunks + task prompt.

    Args:
        task_prompt: the task/question to answer (from eval or PhD prompt files).
        chunks:      ordered list of retrieved payload dicts from retrieve_qdrant_hybrid.
        model:       OpenAI model string.

    Returns:
        Raw LLM response string (may include markdown fences).
    """
    context = format_chunks_for_prompt(chunks)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Retrieved context:\n\n{context}\n\n---\n\nTask: {task_prompt}",
        },
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
    )
    return response.choices[0].message.content


def retrieve_and_generate(task_prompt: str, k: int = 5, model: str = GENERATION_MODEL) -> dict:
    """Full pipeline: retrieve → generate.

    Args:
        task_prompt: task description or question.
        k:           number of chunks to retrieve.
        model:       OpenAI model string.

    Returns:
        Dict with keys: prompt, retrieved_chunk_ids, raw_output.
    """
    chunks = retrieve_qdrant_hybrid(task_prompt, k=k)
    output = generate(task_prompt, chunks, model=model)
    return {
        "prompt":               task_prompt,
        "retrieved_chunk_ids":  [c.get("original_id", "") for c in chunks],
        "raw_output":           output,
    }

def retrieve_and_generate_reranker(task_prompt: str, k: int = 5, model: str = GENERATION_MODEL) -> dict:
    """Full pipeline: retrieve → generate.

    Args:
        task_prompt: task description or question.
        k:           number of chunks to retrieve.
        model:       OpenAI model string.

    Returns:
        Dict with keys: prompt, retrieved_chunk_ids, raw_output.
    """
    chunks = retrieve_qdrant_hybrid_reranked(task_prompt, k=k)
    output = generate(task_prompt, chunks, model=model)
    return {
        "prompt":               task_prompt,
        "retrieved_chunk_ids":  [c.get("original_id", "") for c in chunks],
        "raw_output":           output,
    }


def extract_code_block(text: str) -> str:
    """Strip markdown fences if present; return raw text otherwise.

    Handles ```python ... ``` and plain ``` ... ``` fences.
    """
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()


print("Generation functions defined.")
print(f"Model: {GENERATION_MODEL}")

OpenAI client ready.
Generation functions defined.
Model: gpt-4o


## Smoke Test

One query end-to-end: retrieve → generate → print output.
Verify the retrieved chunks look right before running the full eval loop.

In [23]:
SMOKE_QUERY = qa_pairs_solvers[0]["query"]

result = retrieve_and_generate(SMOKE_QUERY, k=5)

print("Retrieved chunks:")
for i, cid in enumerate(result["retrieved_chunk_ids"], 1):
    print(f"  {i}. {cid}")

print("\n" + "="*60)
print("Generated output:")
print("="*60)
print(result["raw_output"])

Retrieved chunks:
  1. solvers/_common.py:106-110
  2. solvers/poisson.py:69-121
  3. solvers/nonlinear_variable_poisson.py:246-253
  4. solvers/detail/incompressible_euler_bdf_backend.py:181-185
  5. solvers/incompressible_euler.py:13-22

Generated output:
To obtain the physical field along with the assembled matrix, right-hand side pieces, and nullspace metadata for a benchmark with mixed Dirichlet and Neumann elliptic boundary data, you can use the `solve` method from the `PoissonSolver` class. This method is designed to handle Poisson boundary-value problems with mixed boundary conditions and provides all the necessary components for reproducibility. Here's how you can use it:

1. **Prepare Inputs**: You need to provide the following inputs to the `solve` method:
   - `forcing`: A callable or an array representing the forcing term in the Poisson equation.
   - `neu_coeff_func`: A callable or an array representing the Neumann boundary coefficients.
   - `dir_coeff_func`: A callable 

In [28]:
SMOKE_QUERY = qa_pairs_solvers[0]["query"]

result = retrieve_and_generate_reranker(SMOKE_QUERY, k=5)

print("Retrieved chunks:")
for i, cid in enumerate(result["retrieved_chunk_ids"], 1):
    print(f"  {i}. {cid}")

print("\n" + "="*60)
print("Generated output:")
print("="*60)
print(result["raw_output"])

Retrieved chunks:
  1. solvers/poisson.py:69-121
  2. solvers/variable_poisson.py:114-177
  3. solvers/diffusion.py:204-218
  4. solvers/nonlinear_variable_poisson.py:144-165
  5. solvers/nonlinear_variable_poisson.py:224-244

Generated output:
To obtain the physical field along with the assembled matrix, right-hand side pieces, and nullspace metadata for a publication benchmark with mixed Dirichlet and Neumann elliptic boundary data, you can use the `solve` method from either the `PoissonSolver` or `VariablePoissonSolver` classes, depending on whether you have a constant or variable coefficient problem.

Here's a general approach using these solvers:

1. **Choose the Solver**: 
   - Use `PoissonSolver.solve` if you have a constant coefficient problem.
   - Use `VariablePoissonSolver.solve` if you have a variable coefficient problem.

2. **Prepare Inputs**:
   - Define the `forcing` term as a callable or an array.
   - Define the `neu_coeff_func` and `dir_coeff_func` for Neumann and Di

## Validation Stub

Replace the body of `validate()` when Matt/Ram share their validation scripts.

**Expected interface from meeting:**
- Input: path to generated `.py` file + path to ground truth `.py` file
- Output: whether the generated code produces the same results as ground truth

**To plug in:** either replace the function body directly, or call their script as
a subprocess (`subprocess.run(["python", str(their_script), str(gen), str(gt)], ...)`).


In [ ]:
# ── VALIDATION HOOK ────────────────────────────────────────────────────────────
# Replace this stub when Matt/Ram's validation scripts are available.
#
# Their framework (from meeting): run both the ground truth file and the
# generated file, check if they produce the same results for the problem.
#
# Plug in by replacing the body below, or by calling their script as a subprocess:
#
#   result = subprocess.run(
#       ["python", str(VALIDATION_SCRIPT), str(generated_path), str(ground_truth_path)],
#       capture_output=True, text=True,
#   )
#   passed = result.returncode == 0
#
# ── expected return schema ──────────────────────────────────────────────────────
# {
#     "passed":     bool,
#     "difficulty": "easy" | "medium" | "hard",
#     "detail":     str,   # one-line explanation
# }

def validate(generated_path: Path, ground_truth_path: Path) -> dict:
    """Stub. Replace body with PhD validation logic when scripts arrive."""
    raise NotImplementedError(
        "Validation script not yet available. "
        "Replace this stub with Matt/Ram's validate() logic. "
        f"Generated: {generated_path} | Ground truth: {ground_truth_path}"
    )


print("validate() stub defined — replace body when validation scripts arrive.")

## Eval Loop

`run_eval_loop()` is the integration point for the PhD framework.

**Drop their task file here:** `experiments/eval_tasks/tasks.json`

**Expected task JSON schema** (adjust if their format differs):

```json
[
  {
    "id":             "task_easy_01",
    "difficulty":     "easy",
    "prompt":         "Write code to solve a Poisson problem on the unit sphere ...",
    "ground_truth":   "eval_tasks/ground_truth/task_easy_01.py"
  },
  ...
]
```

Generated files are saved to `experiments/eval_outputs/{task_id}.py`.
The loop skips validation if the stub hasn't been replaced yet — it will print a
notice and continue so you can inspect the generated files manually first.

In [ ]:
def run_eval_loop(
    tasks_path:  Path = TASKS_PATH,
    output_dir:  Path = OUTPUT_DIR,
    k:           int  = 5,
    model:       str  = GENERATION_MODEL,
) -> list[dict]:
    """For each task: retrieve → generate → save → validate (if stub replaced).

    Args:
        tasks_path: path to tasks JSON file (PhD format).
        output_dir: directory to write generated .py files.
        k:          retrieval depth.
        model:      OpenAI model string.

    Returns:
        List of result dicts — one per task.
        Each dict: task_id, difficulty, prompt, retrieved_chunk_ids,
                   output_path, validation (None if stub not replaced).
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    if not tasks_path.exists():
        raise FileNotFoundError(
            f"Task file not found: {tasks_path}\n"
            "Drop the PhD task JSON here before running the eval loop."
        )

    with open(tasks_path) as f:
        tasks = json.load(f)

    print(f"Loaded {len(tasks)} tasks from {tasks_path.name}")
    print(f"Output dir: {output_dir}\n")

    results = []
    for task in tasks:
        task_id    = task["id"]
        difficulty = task.get("difficulty", "unknown")
        prompt     = task["prompt"]
        gt_rel     = task.get("ground_truth")
        gt_path    = (EXPERIMENTS_DIR / gt_rel) if gt_rel else None

        print(f"── {task_id} ({difficulty}) ─────────────────────────────")
        print(f"Prompt: {prompt[:100]}...")

        # retrieve → generate
        pipeline_result = retrieve_and_generate(prompt, k=k, model=model)
        code_text = extract_code_block(pipeline_result["raw_output"])

        # save generated file
        output_path = output_dir / f"{task_id}.py"
        output_path.write_text(code_text)
        print(f"Saved → {output_path.name}")
        print(f"Chunks : {pipeline_result['retrieved_chunk_ids']}")

        # validate — skip gracefully if stub not replaced
        validation = None
        if gt_path and gt_path.exists():
            try:
                validation = validate(output_path, gt_path)
                status = "PASS" if validation["passed"] else "FAIL"
                print(f"Validation: {status} — {validation.get('detail', '')}")
            except NotImplementedError:
                print("Validation: skipped (replace validate() stub first)")
        elif gt_path:
            print(f"Validation: skipped (ground truth not found at {gt_path})")
        else:
            print("Validation: skipped (no ground_truth field in task JSON)")

        results.append({
            "task_id":              task_id,
            "difficulty":           difficulty,
            "prompt":               prompt,
            "retrieved_chunk_ids":  pipeline_result["retrieved_chunk_ids"],
            "output_path":          str(output_path),
            "validation":           validation,
        })
        print()

    # summary
    validated = [r for r in results if r["validation"] is not None]
    if validated:
        passed = sum(1 for r in validated if r["validation"]["passed"])
        print(f"Results: {passed}/{len(validated)} passed validation")
        for diff in ["easy", "medium", "hard"]:
            subset = [r for r in validated if r["difficulty"] == diff]
            if subset:
                p = sum(1 for r in subset if r["validation"]["passed"])
                print(f"  {diff:6s}: {p}/{len(subset)}")
    else:
        print(f"Generated {len(results)} files — no validation run yet.")
        print(f"Inspect files in: {output_dir}")

    return results


print("run_eval_loop() defined.")
print(f"\nTask file location  : {TASKS_PATH}")
print(f"Output dir location : {OUTPUT_DIR}")

## Run

Uncomment and run once the PhD task file is in place.
The smoke test above validates the pipeline works before committing to a full run.

In [ ]:
# Uncomment when tasks.json is available at TASKS_PATH.
#
# results = run_eval_loop(k=5, model=GENERATION_MODEL)
#
# Save results log alongside generated files.
# results_path = OUTPUT_DIR / "results.json"
# with open(results_path, "w") as f:
#     json.dump(results, f, indent=2)
# print(f"Results saved → {results_path}")

In [1]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer("jinaai/jina-code-embeddings-0.5b")
print(len(m.encode("test")))

/Users/jordanchambers/public-projects/scientific-codebase-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


896


In [2]:
pip install --upgrade transformers tokenizers --break-system-packages

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached tokenizers-0.23.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 8.9 MB/s  0:00:01 eta 0:00:01
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 8.9 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.5.0
    Uninstalling hf-xet-1.5.0:
      Successfully uninstalled hf-xet-1.5.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [huggingface-hub]
    Found existing installation: tokenizers 0.19.1━━━━━━━━━━━━ 1/4 [huggingface-

In [3]:
from qdrant_client import QdrantClient
c = QdrantClient(host="localhost", port=6333)
c.delete_collection("kernelpack_code")

True